# Pawfolio VoxSOAP - ML Training Notebook

Implementation of **feature/voxsoap-model-integration.md** SS7 (Colab training setup).

## Before you run anything
1. **Runtime > Change runtime type > GPU** (availability varies; the GPU stage verifies it and fails clearly).
2. Upload the ml-training folder (or a zip of it named ML_TRAINING_UPLOAD.zip) so that `/content/ml-training/scripts/*.py` exists. Stage 0 checks this and stops with clear instructions if missing.
3. This notebook is a SCAFFOLD first, then a smoke-test rig: stages print OK / FAILED / SKIPPED(gate) / NOT RUN into the Stage 17 summary. A failure never becomes a claim.

## Hard rails baked into every stage
- Gated datasets (omi-health, common-voice-urdu) are never trainable until a human upgrades the recorded gate in the manifest.
- The official **test split is never** merged into training data.
- Capped subsets (the CONFIG knobs) keep this a smoke test; scale up only after Stage 17 says the small run fully succeeded.
- Never load real Pawfolio/clinic audio or records into this notebook.
- Exports land in `/content/pawfolio/exports`; the laptop target is `D:\\Pawfolio AI\\ml-training\\artifacts\\`.

In [ ]:
# ================================================
# STAGE 0 - payload setup, staging area, stage ledger
# ================================================
import os, sys, io, json, shutil, zipfile, traceback, time
import datetime as dt

try:
    from contextlib import contextmanager
except Exception:
    pass

WORKSPACE = "/content/pawfolio"
ML_DIR = "/content/ml-training"
for p in (WORKSPACE, os.path.join(WORKSPACE, "exports"), os.path.join(WORKSPACE, "data"), os.path.join(WORKSPACE, "reports")):
    os.makedirs(p, exist_ok=True)

# Hugging Face caches must NOT land on the laptop drive (§7.1.4).
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
# TRANSFORMERS_CACHE was dropped (run 6): deprecated in favor of HF_HOME and
# it emits a deprecation warning on every transformers import.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Run-5 lesson (MUST run before ANY transformers import anywhere): transformers
# reads USE_TF/USE_FLAX from the environment at import time and its
# TF-availability gates can end up importing Colab's TensorFlow - whose
# protobuf gencode/runtime pairing breaks when protobuf is downgraded (run 5:
# gencode 6.31.1 vs pinned runtime 5.29.3). This pipeline is pure PyTorch:
# force the torch-only backends here. (TRANSFORMERS_NO_TF is NOT an actual
# flag in transformers 4.48.3 - USE_TF/USE_FLAX are; verified in source.)
# Also: protobuf is deliberately NOT pinned - see requirements-colab.txt.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

LEDGER = {}
RUN_ID = dt.datetime.now().strftime("%Y%m%d-%H%M%S")

@contextmanager
def stage(name, abort=False, note=None):
    """Track every §7.3 stage; failures surface honestly, never silently pass."""
    LEDGER[name] = {"status": "RUNNING", "started": dt.datetime.now().isoformat(timespec="seconds")}
    if note:
        print(f"NOTE[{name}]: {note}")
    print("=" * 76)
    print(f"STAGE {name} - started {LEDGER[name]['started']}")
    print("=" * 76)
    try:
        yield
        LEDGER[name]["status"] = "OK"
        print(f"[STAGE-OK] {name}")
    except Exception as err:
        LEDGER[name]["status"] = "FAILED"
        LEDGER[name]["error"] = f"{type(err).__name__}: {err}"
        LEDGER[name]["trace"] = traceback.format_exc()[-2400:]
        print(f"[STAGE-FAILED] {name} -> {type(err).__name__}: {err}")
        if abort:  # setup stages: continuing is meaningless
            raise
    finally:
        LEDGER[name]["ended"] = dt.datetime.now().isoformat(timespec="seconds")

# --- Stage 0 body: ensure the scripts payload exists (upload or zip extract)
def stage0_payload():
    candidates = [ML_DIR, os.path.join(WORKSPACE, "ml-training")]
    src = next((c for c in candidates if os.path.isdir(os.path.join(c, "scripts"))), None)
    if src is None and os.path.exists("/content/ML_TRAINING_UPLOAD.zip"):
        with zipfile.ZipFile("/content/ML_TRAINING_UPLOAD.zip") as z:
            z.extractall(WORKSPACE)
        src = next((c for c in candidates if os.path.isdir(os.path.join(c, "scripts"))), None)
    if src is None:
        raise RuntimeError(
            "ml-training payload missing. Upload the repository folder ml-training "
            "(locally: D:\\Pawfolio AI\\ml-training) or its zip named "
            "ML_TRAINING_UPLOAD.zip to /content/, then re-run this cell."
        )
    if src != ML_DIR:
        shutil.rmtree(ML_DIR, ignore_errors=True)
        shutil.copytree(src, ML_DIR)
    expected = ["inspect_datasets.py", "prepare_soap_data.py", "prepare_roman_urdu_data.py",
                "train_whisper_lora.py", "train_soap_lora.py", "evaluate_asr.py", "evaluate_soap.py"]
    missing = [f for f in expected if not os.path.exists(os.path.join(ML_DIR, "scripts", f))]
    if missing:
        raise RuntimeError(f"scripts payload incomplete, missing: {missing}")
    os.chdir(ML_DIR)
    manifest_path = os.path.join(ML_DIR, "data_manifest.json")
    if not os.path.exists(manifest_path):
        raise RuntimeError("data_manifest.json missing from the payload - refusing to run without recorded gates.")
    print(f"[STAGE-0-OK] payload present at {ML_DIR}")
    print(f"[STAGE-0-OK] run id {RUN_ID}")
    return ML_DIR

ML = stage0_payload()

In [ ]:
# ================================================
# STAGE 1 - environment and GPU verification (§7.1.3)
# ================================================
import subprocess, importlib

def pip_freeze_line(pkg):
    try:
        mod = importlib.import_module(pkg.split(".")[0])
        return getattr(mod, "__version__", "unknown")
    except Exception as err:
        return f"NOT INSTALLED ({type(err).__name__})"

with stage("1. environment_and_gpu"):
    import torch
    print("python        :", sys.version.split()[0])
    print("pytorch       :", torch.__version__)
    print("cuda built    :", torch.backends.cuda.is_built())
    print("cuda available:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU attached. Runtime > Change runtime type > GPU, then re-run from Stage 0.")
    props = torch.cuda.get_device_properties(0)
    print("gpu model     :", props.name)
    print("gpu memory    :", round(props.total_memory / 1024**3, 2), "GB")
    print("bf16 support  :", torch.cuda.is_bf16_supported())
    # keep versions for the run report (§7.1.3 requires this hardware check)
    STAGE1_ENV = {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "gpu": props.name,
        "gpu_memory_gb": round(props.total_memory / 1024**3, 2),
        "bf16": torch.cuda.is_bf16_supported(),
    }

In [ ]:
# ================================================
# STAGE 2 - install pinned dependencies, SINGLE SOURCE OF TRUTH (§7.3.2)
# Runs 1-2 lesson: the pin list was ALSO hardcoded here and drifted from
# requirements-colab.txt (jiwer), and pip aborts the WHOLE transaction on one
# bad pin, leaving Colab's preinstalled stack behind - so later stages ran on
# an unverified stack without knowing. Now: install from the payload
# requirements file AND assert installed versions match the pins afterwards.
# Run-4 lessons: (a) pins must ship py3.13 wheels - Stage 2 passes
# --only-binary=:all: below, so a pin without wheels fails FAST with
# "No matching distribution" instead of a doomed source build that rolls
# back the whole transaction; (b) Colab preinstalls torchao and peft
# hard-fails on import - removed and asserted absent below, BEFORE anything
# imports peft.
# Run-5 lessons: (c) NEVER pin protobuf on Colab - Colab's TensorFlow is
# compiled against its image's protobuf gencode and a pinned runtime
# downgrade crashes transformers' trainer imports (guarded below, plus a
# [PIN-GUARD] that forbids protobuf in the requirements file); (d)
# USE_TF/USE_FLAX are forced OFF in cell 1 (HF env vars) and at the top of
# every train/eval script BEFORE transformers import, so no TF availability
# probe can ever import Colab's TensorFlow from this pipeline.
# ================================================
with stage("2. install_pinned_dependencies"):
    import subprocess
    import importlib.metadata as _im

    REQS_PATH = os.path.join(ML, "requirements-colab.txt")
    if not os.path.exists(REQS_PATH):
        raise RuntimeError(f"requirements-colab.txt missing from the payload at {REQS_PATH}.")
    with open(REQS_PATH, "r", encoding="utf-8") as fh:
        REQS = [ln.strip() for ln in fh if ln.strip() and not ln.strip().startswith("#")]
    print("[pip] installing EXACTLY the payload pin set:", REQS_PATH)
    for ln in REQS:
        print("   ", ln)

    # Run-4 lesson: sentencepiece source build failure. With exact pins,
    # --prefer-binary CANNOT help (pip may not substitute versions) - insist
    # on pre-compiled wheels instead: a pin without a wheel for this Python
    # then fails FAST ("No matching distribution") instead of a slow doomed
    # source build. Such a gap is fixed in requirements-colab.txt, not by
    # loosening the install here.
    res = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-warn-script-location",
                          "--only-binary=:all:", "-r", REQS_PATH],
                         capture_output=True, text=True, timeout=3000)
    if res.returncode != 0:
        print("[pip] STDOUT tail:", (res.stdout or "")[-1200:])
        print("[pip] STDERR tail:", (res.stderr or "")[-2400:])
        raise RuntimeError("pip install of the requirements file failed - fix the PINS (never install unpinned replacements) and re-run Stage 2.")

    # Post-install drift check: pip success alone does not prove the pinned
    # version is what is actually present (Colab preinstalls its own stack).
    EXPECTED = {}
    for ln in REQS:
        if "==" in ln:
            name, want = ln.split("==", 1)
            EXPECTED[name.strip()] = want.strip()
    got, drift = {}, []
    for name, want in EXPECTED.items():
        try:
            actual = _im.version(name)
        except Exception:
            raise RuntimeError(f"[PIN-DRIFT] '{name}' is not installed after pip succeeded - investigate before continuing.")
        got[name] = actual
        if actual != want:
            drift.append(f"{name}: pinned {want}, installed {actual}")
    if drift:
        raise RuntimeError("[PIN-DRIFT] " + " | ".join(drift) + " - re-run Stage 2 after fixing; do NOT continue on a different stack.")

    # Run-4 fix: Colab PREINSTALLS torchao (0.10.0 on the current image) and
    # peft's LoRA dispatch calls is_torchao_available(), which RAISES a hard
    # ImportError for torchao<=0.16 even though this pipeline never imports
    # torchao - Stage 9/12 crash inside get_peft_model() (run 4 traceback).
    # Uninstall it here, before any peft import, then assert it is truly
    # gone. We never use torchao; bitsandbytes is the only quantization stack
    # this pipeline needs, per requirements-colab.txt.
    import importlib.util as _iu
    if "torchao" in EXPECTED:
        raise RuntimeError("[PIN-GUARD] torchao must NOT be pinned in requirements-colab.txt - remove the pin; Stage 2 uninstalls the runtime's preinstalled copy instead.")
    ta_un = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
                           capture_output=True, text=True, timeout=600)
    if ta_un.returncode != 0:
        print("[pip] torchao uninstall STDERR tail:", (ta_un.stderr or "")[-800:])
        raise RuntimeError("pip uninstall of torchao failed - peft 0.18.1 cannot be imported while torchao<=0.16 is present.")
    if _iu.find_spec("torchao") is not None:
        raise RuntimeError("[BLOCKED-STACK] torchao is still importable after uninstall - peft raises a hard ImportError for torchao<=0.16; open a fresh runtime and re-run Stage 2.")
    try:
        _ta_left = _im.version("torchao")
        raise RuntimeError(f"[BLOCKED-STACK] torchao {_ta_left} still has dist metadata after uninstall - re-run Stage 2 in a fresh runtime.")
    except _im.PackageNotFoundError:
        pass
    print("[STACK-OK] torchao absent - peft LoRA dispatch will not hit the run-4 ImportError")

    # Run-5 fix: protobuf is RUNTIME-OWNED on Colab. Colab's TensorFlow is
    # compiled against its image's protobuf gencode; pip-installing an older
    # protobuf PIN (5.29.3 in run 5) downgraded the RUNTIME and TF crashed on
    # import inside transformers' Trainer integration path
    # (gencode 6.31.1 vs runtime 5.29.3). Registry-verified: no pinned
    # package requires protobuf as a runtime dep (extras only), so we never
    # pin or install it - the preinstalled copy stays and must keep pairing.
    if "protobuf" in EXPECTED:
        raise RuntimeError("[PIN-GUARD] protobuf must NOT be pinned on Colab - Colab's TensorFlow gencode must pair with the runtime's protobuf major. Remove the pin; the preinstalled copy stays (run-5 lesson).")
    try:
        pb_runtime = _im.version("protobuf")
    except _im.PackageNotFoundError:
        pb_runtime = None
    print(f"[STACK-OK] protobuf left at runtime version {pb_runtime or '(absent)'} - never pinned or installed by Stage 2")

    versions = dict(got)
    versions["stack_fixes"] = [
        "torchao was preinstalled and uninstalled by Stage 2 (peft 0.18.1 raises an unconditional ImportError for torchao<=0.16; pipeline uses bitsandbytes)",
        f"protobuf left at the runtime's preinstalled version ({pb_runtime or 'absent'}) - NOT pinned, NOT installed by Stage 2 (TF gencode/runtime pairing is runtime-owned; pinning it crashed transformers trainer imports in run 5)",
    ]
    versions["USE_TF"] = os.environ.get("USE_TF", "0")
    versions["USE_FLAX"] = os.environ.get("USE_FLAX", "0")
    versions.update(STAGE1_ENV)
    with open(os.path.join(WORKSPACE, "reports", f"environment_versions_{RUN_ID}.json"), "w") as fh:
        json.dump(versions, fh, indent=2)
    print("[STAGE-2-OK] installed versions verified against pins:", versions)

In [ ]:
# ================================================
# STAGE 3 - workspace / cache / disk availability (§7.3.3)
# ================================================
with stage("3. workspace_and_disk"):
    import shutil
    # Stage 0 sets the HF cache env vars, but the directory is created lazily by
    # the HF libraries - probing it before their first use raised FileNotFound
    # (run 2). Create it explicitly, then measure.
    os.makedirs("/content/hf_cache", exist_ok=True)
    usage = shutil.disk_usage("/content")
    free_gb = round(usage.free / 1024**3, 2)
    free_hf = round(shutil.disk_usage("/content/hf_cache").free / 1024**3, 2)
    print(f"/content free: {free_gb} GB | HF cache free: {free_hf} GB")
    if free_gb < 15:
        print("[WARN] less than 15 GB free - smoke run should still fit, but no full corpus processing is allowed.")
    stage_disk = {"free_gb": free_gb, "hf_cache_free_gb": free_hf}
    json.dump(stage_disk, open(os.path.join(WORKSPACE, "reports", f"disk_{RUN_ID}.json"), "w"), indent=2)
    print("[STAGE-3-OK]")

In [ ]:
# ================================================
# STAGE 4 - dataset gates: cards, configs, splits, revisions (§7.3.4)
# ================================================
with stage("4. dataset_inspection_and_manifest"):
    import subprocess
    out_manifest = os.path.join(WORKSPACE, "reports", f"data_manifest_runtime_{RUN_ID}.json")
    cmd = [sys.executable, os.path.join(ML, "scripts", "inspect_datasets.py"),
           "--manifest-in", os.path.join(ML, "data_manifest.json"),
           "--manifest-out", out_manifest]
    res = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    print(res.stdout[-8000:] if res.stdout else "(no stdout)")
    if res.returncode != 0:
        print(res.stderr[-4000:])
        LEDGER["4. dataset_inspection_and_manifest"]["status"] = "FAILED"
        raise RuntimeError("dataset inspection failed - stages below must NOT train on unaudited sources.")
    # load gate status for later stages
    with open(out_manifest, "r", encoding="utf-8") as fh:
        RUNTIME_MANIFEST = json.load(fh)
    APPROVED_IDS = {s["dataset_id"] for s in RUNTIME_MANIFEST.get("sources", []) if s.get("license_status") in ("APPROVED", "APPROVED_OPTIONAL")}
    print("[APPROVED sources]:", sorted(APPROVED_IDS))
    print("[STAGE-4-OK]")

In [ ]:
# ================================================
# STAGE 5 - validate sample records of every approved source (§7.3.5)
# ================================================
# Phase-3 knobs (run-8 mandate: fail-safe toggles let a free-T4 session run
# all components or stop/crash out and resume with a subset in a new session;
# every skipped component is honest-reported in the Stage 17 ledger).
RUN_ASR = True
RUN_SOAP = True
RUN_MODEL_C = True

CONFIG_KNOBS = {
    # Phase-3 full-scale caps (§7.1.7). The run-8 stream scan cost scales
    # with the cap - ur_pk ~122 raw rows per 120 kept (near-perfect source);
    # 1200 train rows needs scanning ~1220+ raw FLEURS rows per split.
    "ASR_FLEURS_UR_TRAIN": 1200,
    "ASR_FLEURS_UR_VAL": 200,
    "ASR_FLEURS_UR_TEST": 100,
    "ASR_FLEURS_EN_TRAIN": 300,
    "ASR_FLEURS_EN_VAL": 60,
    "ASR_FLEURS_EN_TEST": 60,
    "SOAP_TRAIN": 400,
    "SOAP_VAL": 100,
    "SOAP_TEST": 100,
    "SOAP_MAX_SEQ_TOKENS": 1024,
    "SOAP_MAX_NEW_TOKENS": 512,
    "SOAP_EPOCHS": 2,
    "ASR_EPOCHS": 3,
    "ASR_LR": 0.00008,
    "ASR_LORA_R": 16,
    "ASR_WARMUP": 0.05,
    "ASR_EVAL_STEPS": 50,
    "ASR_TARGETS": "q_proj,k_proj,v_proj,o_proj",
    "ROMAN_URDU_TRAIN": 2000,
    "ROMAN_URDU_VAL": 300,
    "ROMAN_URDU_TEST": 300,
    # Run-10 remediation (fp16-AMP at lr 5e-4 collapsed to sentinel output):
    # gentler LR, earlier eval points, PURE-fp32 trainer.
    "ROMAN_LR": 0.0001,
    "ROMAN_EVAL_STEPS": 50,
    # Run-11 remediation (fp32+lr1e-4 also stalled at loss~ln(vocab) with
    # sentinel-headed outputs): ALIGN to mt5's span-corruption format - targets
    # get the '<extra_id_0>' prefix (eval/decode strips it), wider LoRA on the
    # T5 attention q/k/v/o planes, and more optimizer steps (8 epochs).
    "ROMAN_EPOCHS": 8,
    "ROMAN_TARGETS": "q,k,v,o",
}
SEED = 20260124

with stage("5. sample_record_validation"):
    from datasets import load_dataset

    def peek_stream(dataset_id, split, config=None, want_fields=("transcription", "sentence", "output", "target", "soap", "input")):
        stream = load_dataset(dataset_id, config, split=split, streaming=True) if config else load_dataset(dataset_id, split=split, streaming=True)
        first = next(iter(stream), None)
        if first is None:
            raise RuntimeError(f"{dataset_id} [{split}] produced no records (check id/split/name against the runtime manifest).")
        keys = list(first.keys())
        print(f"[PEEK] {dataset_id}{('/' + str(config)) if config else ''} [{split}] fields={keys[:14]}")
        # chat-style corpora expose ONLY a 'messages' column (seen live for
        # medscribe-soap-712 in the 2026-10-24 smoke run): map roles honestly.
        if isinstance(first.get("messages"), list) and first.get("messages"):
            msgs = first["messages"]
            roles = [str(m.get("role")) for m in msgs if isinstance(m, dict)]
            user_txt = " ".join(str(m.get("content")) for m in msgs if isinstance(m, dict) and str(m.get("role", "")).lower() == "user")
            asst_txt = " ".join(str(m.get("content")) for m in msgs if isinstance(m, dict) and str(m.get("role", "")).lower() == "assistant")
            if not user_txt.strip() or not asst_txt.strip():
                print(f"[PEEK-FAIL-LOUD] {dataset_id}: 'messages' lacks a user+assistant pair; roles={roles}")
                raise RuntimeError(f"{dataset_id} [{split}] messages format needs agent extension - NOT silently skipped.")
            print(f"[PEEK] chat format: roles={roles}")
            print(f"[PEEK] user preview  : {user_txt[:120]!r}")
            print(f"[PEEK] assistant preview: {asst_txt[:120]!r}")
            return {"keys": keys, "text_field": "messages", "format": "chat_messages", "roles": roles}
        text_key = next((k for k in want_fields if k in keys and isinstance(first[k], str) and first[k].strip()), None)
        if text_key is None:
            text_key = next((k for k in keys if isinstance(first.get(k), str)), None)
        if not text_key or not str(first[text_key]).strip():
            raise RuntimeError(f"{dataset_id} [{split}] has no usable text field; found keys {keys}")
        preview = str(first[text_key])[:120].replace("\n", " / ")
        print(f"[PEEK] text field '{text_key}' sample: {preview}")
        return {"keys": keys, "text_field": text_key}

    PEEK_REPORT = {}
    if "google/fleurs" in APPROVED_IDS:
        PEEK_REPORT["google/fleurs ur_pk train"] = peek_stream("google/fleurs", "train", "ur_pk")
    else:
        print("[GATE-SKIP] google/fleurs not APPROVED at this revision - ASR stages will be skipped.")
    if "Tushar9802/medscribe-soap-712" in APPROVED_IDS:
        PEEK_REPORT["soap"] = peek_stream("Tushar9802/medscribe-soap-712", "train")
    else:
        print("[GATE-SKIP] soap corpus not APPROVED - SOAP stages will be skipped.")
    if "Mavkif/Roman-Urdu-Parl-split" in APPROVED_IDS:
        PEEK_REPORT["roman_urdu"] = peek_stream("Mavkif/Roman-Urdu-Parl-split", "train")
    else:
        print("[GATE-SKIP] roman urdu corpus not APPROVED - Stage 14 will be skipped.")
    json.dump(PEEK_REPORT, open(os.path.join(WORKSPACE, "reports", f"sample_fields_{RUN_ID}.json"), "w"), indent=2)
    print("[STAGE-5-OK]")

In [ ]:
# ================================================
# STAGE 6 - stream, decode and validate audio rows (§7.3.6)
# Version-proof audio decode: streaming datasets hand audio back differently
# across majors (dict vs lazy AudioDecoder objects). Cast the column to
# Audio(decode=False) - yields {bytes, path} on EVERY datasets version - then
# decode the bytes with soundfile (pinned; no torchcodec needed). Skip reasons
# are counted SEPARATELY (run-2 lesson: 267 uniform skips hid the true cause)
# and the first failing row is printed verbatim for diagnosis.
# ================================================
import io

import librosa as _lb
import numpy as np
import soundfile as sf
from datasets import load_dataset, Audio as _Audio

TARGET_SR = 16000

def stream_audio_raw(dataset_id, config, split):
    stream = load_dataset(dataset_id, config, split=split, streaming=True)
    try:
        stream = stream.cast_column("audio", _Audio(decode=False))
    except Exception as cast_err:
        print(f"[DECODE-WARN] cast_column(audio, decode=False) failed ({cast_err}); continuing with the default stream.")
    return stream

def decode_audio_field(audio_field):
    """Returns (float32 array | None, sampling_rate | None). Never raises."""
    if isinstance(audio_field, dict):
        raw = audio_field.get("bytes")
        if raw:
            try:
                data, sr = sf.read(io.BytesIO(raw), dtype="float32")
            except Exception:
                return None, None
            data = np.asarray(data, dtype=np.float32)
            if data.ndim == 2:
                data = data.mean(axis=1).astype(np.float32)
            return data, int(audio_field.get("sampling_rate") or sr or 0)
        arr = audio_field.get("array")
        if arr is not None:
            return np.asarray(arr, dtype=np.float32), int(audio_field.get("sampling_rate") or 0)
    # Best-effort fallback for lazy decoder objects (only reached if the cast
    # above failed - e.g. an unexpected datasets version).
    for meth_name in ("get_all_samples", "decode"):
        fn = getattr(audio_field, meth_name, None)
        if callable(fn):
            try:
                samples = fn()
            except Exception:
                continue
            data = getattr(samples, "data", getattr(samples, "audio", None))
            sr = getattr(samples, "sample_rate", 0)
            if data is not None:
                try:
                    arr = np.asarray(data, dtype=np.float32)
                    if arr.ndim >= 2:
                        arr = arr.mean(axis=0).astype(np.float32)
                    return arr, int(sr or 0)
                except Exception:
                    continue
    return None, None

def decode_check(dataset_id, config, split, cap=8):
    stream = stream_audio_raw(dataset_id, config, split)
    ok = 0
    reasons = {}
    srs = []

    def bump(reason, row, audio_field):
        reasons[reason] = reasons.get(reason, 0) + 1
        if reasons[reason] == 1:  # print each reason's FIRST occurrence verbatim
            keys = list(row.keys()) if isinstance(row, dict) else "?"
            print(f"[DECODE-DIAG] first '{reason}' row: keys={keys}")
            if audio_field is not None:
                print(f"[DECODE-DIAG] audio type={type(audio_field).__name__} repr={str(audio_field)[:180]}")

    for row in stream:
        if ok >= cap:
            break
        sent = str(row.get("transcription") or row.get("sentence") or row.get("text") or "").strip()
        if not sent:
            bump("empty_transcript", row, None)
            continue
        audio_field = row.get("audio")
        arr, sr = decode_audio_field(audio_field)
        if arr is None or len(arr) == 0 or not sr:
            bump("decode_failed", row, audio_field)
            continue
        if sr != TARGET_SR:  # resample instead of ASSUMING 16k (§5 note)
            try:
                arr = _lb.resample(arr, orig_sr=sr, target_sr=TARGET_SR)
                sr = TARGET_SR
            except Exception:
                bump("resample_failed", row, audio_field)
                continue
        seconds = len(arr) / float(TARGET_SR)
        if seconds < 0.2 or seconds > 60:
            bump("bad_length", row, audio_field)
            continue
        srs.append(sr)
        ok += 1
        if ok == 1:
            print(f"[DECODE] first ok row: sr={sr}, seconds={seconds:.2f}, transcript='{sent[:80]}'")
    print(f"[DECODE] {dataset_id}/{config} [{split}] ok={ok} skip_reasons={reasons} sr_seen={sorted(set(srs))}")
    if ok == 0:
        raise RuntimeError(f"audio decode produced zero usable rows (skip_reasons={reasons}) - fix before training.")
    return ok, reasons, sorted(set(srs))

if not RUN_ASR:
    LEDGER["6. audio_stream_decode_validation"] = {"status": "SKIPPED(toggle)", "error": "RUN_ASR=False (Roman-only session)"}
    print("[TOGGLE-SKIP] audio decode validation skipped (RUN_ASR=False).")
else:
    with stage("6. audio_stream_decode_validation"):
        decode_stats = {}
        if "google/fleurs" in APPROVED_IDS:
            decode_stats["ur_pk_valid"] = decode_check("google/fleurs", "ur_pk", "validation")
            decode_stats["en_us_valid"] = decode_check("google/fleurs", "en_us", "validation")
            json.dump(decode_stats, open(os.path.join(WORKSPACE, "reports", f"audio_decode_{RUN_ID}.json"), "w"), indent=2)
        else:
            LEDGER["6. audio_stream_decode_validation"]["status"] = "SKIPPED(gate)"
            print("[GATE-SKIP] no approved audio source available for decode validation.")
        print("[STAGE-6-OK]")

In [ ]:
# ================================================
# STAGE 7 - capped, reproducible ASR training data WITHOUT test leakage (§7.3.7)
# ================================================
try:
    import io

    import librosa as _lb
    import soundfile as sf
    from datasets import load_dataset, Dataset, concatenate_datasets, Features, Audio, Value
    import numpy as np
except Exception as import_err:  # honest labeling: record to the ledger, then re-raise
    LEDGER["7. asr_capped_training_prep"] = {"status": "FAILED", "error": f"import failed: {type(import_err).__name__}: {import_err}"}
    raise

TARGET_SR = 16000
# Run-6 hardening: canonical MINIMAL store schema - the saved dataset keeps
# ONLY "audio" (16 kHz mono float32 arrays) and "text". All extra FLEURS
# columns (id/num_samples/path/raw_transcription/gender/lang_id/...) AND our
# own convenience provenance columns are stripped from the store; provenance
# (source_row_ids per split) is preserved in the Stage 7 JSON report instead,
# so a stray residual column can never surface as a feature or target.
ASM_FEATURES = Features({
    "audio": Audio(sampling_rate=TARGET_SR),
    "text": Value("string"),
})
# Whisper stability + smoke speed caps (§7.3.6, run-6 mandate):
MIN_TEXT_CHARS = 2
MIN_SECONDS = 0.5   # < 8,000 samples @ 16 kHz
MAX_SECONDS = 20.0  # > 320,000 samples @ 16 kHz
SILENCE_EPS = 1e-4

def stream_audio_raw(config, split):
    stream = load_dataset("google/fleurs", config, split=split, streaming=True)
    try:
        stream = stream.cast_column("audio", Audio(decode=False))
    except Exception as cast_err:
        print(f"[PREP-WARN] cast_column(audio, decode=False) failed ({cast_err}); continuing with the default stream.")
    return stream

def decode_audio_field(audio_field):
    """Version-proof decode: {bytes,path} -> soundfile; dict-with-array passthrough."""
    if isinstance(audio_field, dict):
        raw = audio_field.get("bytes")
        if raw:
            try:
                data, sr = sf.read(io.BytesIO(raw), dtype="float32")
            except Exception:
                return None, None
            data = np.asarray(data, dtype=np.float32)
            if data.ndim == 2:
                data = data.mean(axis=1).astype(np.float32)
            return data, int(audio_field.get("sampling_rate") or sr or 0)
        arr = audio_field.get("array")
        if arr is not None:
            return np.asarray(arr, dtype=np.float32), int(audio_field.get("sampling_rate") or 0)
    # Best-effort for lazy decoder objects (only if the cast above failed).
    for meth_name in ("get_all_samples", "decode"):
        fn = getattr(audio_field, meth_name, None)
        if callable(fn):
            try:
                samples = fn()
            except Exception:
                continue
            data = getattr(samples, "data", getattr(samples, "audio", None))
            sr = getattr(samples, "sample_rate", 0)
            if data is not None:
                try:
                    arr = np.asarray(data, dtype=np.float32)
                    if arr.ndim >= 2:
                        arr = arr.mean(axis=0).astype(np.float32)
                    return arr, int(sr or 0)
                except Exception:
                    continue
    return None, None

def prep_bump(reasons, reason, row, audio_field):
    reasons[reason] = reasons.get(reason, 0) + 1
    if reasons[reason] == 1:  # first occurrence of each reason, printed verbatim
        keys = list(row.keys()) if isinstance(row, dict) else "?"
        print(f"[PREP-DIAG] first '{reason}' row: keys={keys}")
        if audio_field is not None:
            print(f"[PREP-DIAG] audio type={type(audio_field).__name__} repr={str(audio_field)[:180]}")

def build_split(config, split, cap):
    """Stream, decode, normalize, and CLEAN a capped split (run-6 mandate).

    Canonical store columns: audio (16 kHz mono float32) + text, nothing
    else. Exact filter accounting per split: raw_rows, dropped_empty_text,
    dropped_duration_outlier, dropped_silent_or_corrupt (decode/resample
    failures fold here - they ARE corrupt inputs), kept_clean_rows.
    """
    stream = stream_audio_raw(config, split)
    rows = []
    stats = {
        "raw_rows": 0,
        "dropped_empty_text": 0,
        "dropped_duration_outlier": 0,
        "dropped_silent_or_corrupt": 0,
        "kept_clean_rows": 0,
        "source_row_ids": [],
    }

    for row in stream:
        if stats["kept_clean_rows"] >= cap:
            break
        stats["raw_rows"] += 1
        text = str(row.get("transcription") or row.get("sentence") or row.get("text") or "").strip()
        if len(text) < MIN_TEXT_CHARS:
            prep_bump(stats, "dropped_empty_text", row, None)
            continue
        audio_field = row.get("audio")
        arr, sr = decode_audio_field(audio_field)
        if arr is None or len(arr) == 0 or not sr:
            prep_bump(stats, "dropped_silent_or_corrupt", row, audio_field)
            continue
        arr = np.asarray(arr, dtype=np.float32)
        if arr.ndim > 1:  # stereo/multi-channel -> mono (never assumed)
            arr = arr.mean(axis=-1)
        if sr != TARGET_SR:  # never assume 16k from the name alone
            try:
                arr = _lb.resample(arr, orig_sr=sr, target_sr=TARGET_SR)
            except Exception:
                prep_bump(stats, "dropped_silent_or_corrupt", row, audio_field)
                continue
        arr = np.ascontiguousarray(arr.reshape(-1), dtype=np.float32)
        n_samples = len(arr)
        if n_samples < int(MIN_SECONDS * TARGET_SR) or n_samples > int(MAX_SECONDS * TARGET_SR):
            prep_bump(stats, "dropped_duration_outlier", row, audio_field)
            continue
        if float(np.max(np.abs(arr))) < SILENCE_EPS:
            prep_bump(stats, "dropped_silent_or_corrupt", row, audio_field)
            continue
        rows.append({
            "audio": {"array": arr, "sampling_rate": TARGET_SR},
            "text": text,
        })
        stats["kept_clean_rows"] += 1
        stats["source_row_ids"].append(str(row.get("id") or row.get("audio_id") or f"{config}-{split}-{stats['kept_clean_rows']}"))

    brief = {k: v for k, v in stats.items() if k != "source_row_ids"}
    print(f"[PREP] {config} [{split}] {json.dumps(brief)} cap={cap}")
    if not rows:
        return None, stats
    # Explicit Audio(sampling_rate=16000) feature: audio is stored as properly
    # encoded PCM, decoded uniformly by datasets 2.x AND >=3.3 readers.
    cols = {
        "audio": [r["audio"] for r in rows],
        "text": [r["text"] for r in rows],
    }
    return Dataset.from_dict(cols, features=ASM_FEATURES), stats

if not RUN_ASR:
    LEDGER["7. asr_capped_training_prep"] = {"status": "SKIPPED(toggle)", "error": "RUN_ASR=False (Roman-only session)"}
    print("[TOGGLE-SKIP] ASR data prep skipped (RUN_ASR=False).")
else:
    with stage("7. asr_capped_training_prep"):
        train_parts, val_parts, test_parts, prep_stats = [], [], [], {}
        if "google/fleurs" in APPROVED_IDS:
            for cfg, train_cap, val_cap, test_cap in (
                ("ur_pk", CONFIG_KNOBS["ASR_FLEURS_UR_TRAIN"], CONFIG_KNOBS["ASR_FLEURS_UR_VAL"], CONFIG_KNOBS["ASR_FLEURS_UR_TEST"]),
                ("en_us", CONFIG_KNOBS["ASR_FLEURS_EN_TRAIN"], CONFIG_KNOBS["ASR_FLEURS_EN_VAL"], CONFIG_KNOBS["ASR_FLEURS_EN_TEST"]),
            ):
                tr, st1 = build_split(cfg, "train", train_cap)
                va, st2 = build_split(cfg, "validation", val_cap)
                te, st3 = build_split(cfg, "test", test_cap)
                prep_stats[cfg] = {"train": st1, "validation": st2, "test": st3}
                if tr: train_parts.append(tr)
                if va: val_parts.append(va)
                if te: test_parts.append(te)
        else:
            LEDGER["7. asr_capped_training_prep"]["status"] = "SKIPPED(gate)"
            print("[GATE-SKIP] google/fleurs is not approved at this revision; ASR training data NOT built.")

        if train_parts and val_parts and test_parts:
            train_ds = concatenate_datasets(train_parts) if len(train_parts) > 1 else train_parts[0]
            val_ds = concatenate_datasets(val_parts) if len(val_parts) > 1 else val_parts[0]
            test_ds = concatenate_datasets(test_parts) if len(test_parts) > 1 else test_parts[0]
            from datasets import DatasetDict
            asr_store = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})
            # TEST NEVER enters the trainer. Saved separately and only referenced by eval stages.
            asr_store.save_to_disk(os.path.join(WORKSPACE, "data", "asr_store"))
            json.dump(prep_stats, open(os.path.join(WORKSPACE, "reports", f"asr_prep_{RUN_ID}.json"), "w"), indent=2)
            print(f"[STAGE-7-OK] train={len(train_ds)} validation={len(val_ds)} test={len(test_ds)} (test saved untouched)")
        elif LEDGER["7. asr_capped_training_prep"]["status"] == "RUNNING":
            LEDGER["7. asr_capped_training_prep"]["status"] = "FAILED"
            raise RuntimeError("ASR prep produced no usable train/val/test sets (see skips above).");

In [ ]:
# ================================================
# STAGE 8 - evaluate the BASE ASR checkpoint BEFORE fine-tuning (§7.3.8)
# ================================================
if not RUN_ASR:
    LEDGER["8. asr_base_eval"] = {"status": "SKIPPED(toggle)", "error": "RUN_ASR=False (Roman-only session)"}
    print("[TOGGLE-SKIP] base ASR evaluation skipped (RUN_ASR=False).")
else:
    with stage("8. asr_base_eval"):
        import subprocess
        val_path = os.path.join(WORKSPACE, "data", "asr_store", "validation")
        base_report = os.path.join(WORKSPACE, "reports", f"asr_base_eval_{RUN_ID}.json")
        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "evaluate_asr.py"),
                              "--eval-dataset", val_path,
                              "--base-model", "openai/whisper-small",
                              "--sample-cap", "25", "--report", base_report],
                             capture_output=True, text=True, timeout=3600)
        print(res.stdout[-6000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-3000:])
            LEDGER["8. asr_base_eval"]["status"] = "FAILED"
            raise RuntimeError("base ASR evaluation failed.")
        ASR_BASE_REPORT = base_report
        print("[STAGE-8-OK]")

In [ ]:
# ================================================
# STAGE 9 - fine-tune the ASR adapter (§7.3.9)
# ================================================
TRAIN_ASR = RUN_ASR and ("google/fleurs" in APPROVED_IDS or LEDGER["7. asr_capped_training_prep"]["status"] == "OK")

if TRAIN_ASR:
    with stage("9. asr_lora_train", note="phase-3 scale: r=16 on q/k/v/o, 3 epochs, lr 8e-5; eval_steps=50 now fires"):
        import subprocess
        train_path = os.path.join(WORKSPACE, "data", "asr_store", "train")
        val_path = os.path.join(WORKSPACE, "data", "asr_store", "validation")
        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "train_whisper_lora.py"),
                              "--train-dataset", train_path,
                              "--val-dataset", val_path,
                              "--base-model", "openai/whisper-small",
                              "--epochs", str(CONFIG_KNOBS["ASR_EPOCHS"]),
                              "--learning-rate", str(CONFIG_KNOBS["ASR_LR"]),
                              "--lora-r", str(CONFIG_KNOBS["ASR_LORA_R"]),
                              "--lora-targets", CONFIG_KNOBS["ASR_TARGETS"],
                              "--eval-steps", str(CONFIG_KNOBS["ASR_EVAL_STEPS"]),
                              "--warmup-ratio", str(CONFIG_KNOBS["ASR_WARMUP"]),
                              "--model-scale", "full",
                              "--export-dir", os.path.join(WORKSPACE, "exports", "asr"),
                              "--out-name", f"whisper-lora-ur-{RUN_ID}"],
                             capture_output=True, text=True, timeout=14400)
        print(res.stdout[-8000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-4000:])
            LEDGER["9. asr_lora_train"]["status"] = "FAILED"
            raise RuntimeError("ASR LoRA training failed - adapter NOT saved.")
        ASR_ADAPTER_DIR = os.path.join(WORKSPACE, "exports", "asr", f"whisper-lora-ur-{RUN_ID}")
        print("[STAGE-9-OK]")
elif not RUN_ASR:
    LEDGER["9. asr_lora_train"] = {"status": "SKIPPED(toggle)", "error": "RUN_ASR=False (Colab quota safeguard)"}
    print("[TOGGLE-SKIP] ASR training skipped (RUN_ASR=False).")
else:
    LEDGER["9. asr_lora_train"] = {"status": "SKIPPED(gate)", "error": "no approved ASR source"}
    print("[GATE-SKIP] ASR training skipped (no approved source).")

In [ ]:
# ================================================
# STAGE 10 - evaluate the FINE-TUNED adapter on the SAME held-out data (§7.3.10)
# ================================================
if TRAIN_ASR and not LEDGER["9. asr_lora_train"].get("error"):
    with stage("10. asr_adapter_eval"):
        import subprocess
        val_path = os.path.join(WORKSPACE, "data", "asr_store", "validation")
        tune_report = os.path.join(WORKSPACE, "reports", f"asr_adapter_eval_{RUN_ID}.json")
        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "evaluate_asr.py"),
                              "--eval-dataset", val_path,
                              "--base-model", "openai/whisper-small",
                              "--adapter", os.path.join(WORKSPACE, "exports", "asr", f"whisper-lora-ur-{RUN_ID}"),
                              "--sample-cap", "25", "--report", tune_report],
                             capture_output=True, text=True, timeout=3600)
        print(res.stdout[-6000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-3000:])
            LEDGER["10. asr_adapter_eval"]["status"] = "FAILED"
            raise RuntimeError("fine-tuned ASR evaluation failed.")
        try:
            base = json.load(open(ASR_BASE_REPORT))
            tuned = json.load(open(tune_report))
            print(f"[COMPARISON] base WER {base['wer']} -> tuned WER {tuned['wer']} (same normalization rules)")
        except Exception as err:
            print(f"[COMPARISON] could not compare reports: {err}")
        print("[STAGE-10-OK]")
else:
    LEDGER["10. asr_adapter_eval"] = {"status": "NOT RUN", "error": "stage 9 did not produce an adapter"}
    print("[NOT-RUN] adapter evaluation skipped (stage 9 failed or gated).")

In [ ]:
# ================================================
# STAGE 11 - prepare approved SOAP examples and validate schema (§7.3.11)
# ================================================
SOAP_DATASET_OK = "Tushar9802/medscribe-soap-712" in APPROVED_IDS
if not RUN_SOAP:
    LEDGER["11. soap_prep"] = {"status": "SKIPPED(toggle)", "error": "RUN_SOAP=False (Roman-only session)"}
    print("[TOGGLE-SKIP] SOAP preparation skipped (RUN_SOAP=False).")
elif SOAP_DATASET_OK:
    with stage("11. soap_prep"):
        import subprocess
        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "prepare_soap_data.py"),
                              "--dataset", "Tushar9802/medscribe-soap-712",
                              "--manifest", out_manifest,
                              "--out-dir", os.path.join(WORKSPACE, "data", "soap"),
                              "--max-per-split", str(CONFIG_KNOBS["SOAP_TRAIN"]),
                              "--max-test-rows", str(CONFIG_KNOBS["SOAP_TEST"]),
                              "--seed", str(SEED)],
                             capture_output=True, text=True, timeout=3600)
        print(res.stdout[-7000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-3000:])
            LEDGER["11. soap_prep"]["status"] = "FAILED"
            raise RuntimeError("SOAP preparation failed.")
        soap_report_path = os.path.join(WORKSPACE, "data", "soap", "soap_prep_report.json")
        soap_prep = json.load(open(soap_report_path))
        print("[STAGE-11-OK] fields:", soap_prep["discovered_fields"], "| accepted:", soap_prep["accepted"], "| rejected:", soap_prep["rejected"])
else:
    LEDGER["11. soap_prep"] = {"status": "SKIPPED(gate)", "error": "SOAP dataset not approved"}
    print("[GATE-SKIP] SOAP prep skipped (unapproved).")

In [ ]:
# ================================================
# STAGE 12 - fine-tune the SOAP model (assistant-token loss) + draft test rows (§7.3.12)
# ================================================
RUN_SOAP_STAGE = RUN_SOAP and SOAP_DATASET_OK and LEDGER["11. soap_prep"]["status"] == "OK"
if RUN_SOAP_STAGE:
    with stage("12. soap_lora_train", note="phase-3 scale: 2 epochs, seq cap 1024, drafts over ALL test rows at <=512 new tokens"):
        import subprocess
        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "train_soap_lora.py"),
                              "--train-jsonl", os.path.join(WORKSPACE, "data", "soap", "soap_train.jsonl"),
                              "--val-jsonl", os.path.join(WORKSPACE, "data", "soap", "soap_validation.jsonl"),
                              "--base-model", "Qwen/Qwen2.5-3B-Instruct",
                              "--epochs", str(CONFIG_KNOBS["SOAP_EPOCHS"]),
                              "--max-samples", str(CONFIG_KNOBS["SOAP_TRAIN"]),
                              "--max-seq-tokens", str(CONFIG_KNOBS["SOAP_MAX_SEQ_TOKENS"]),
                              "--model-scale", "full",
                              "--export-dir", os.path.join(WORKSPACE, "exports", "soap"),
                              "--out-name", f"soap-lora-{RUN_ID}"],
                             capture_output=True, text=True, timeout=21600)
        print(res.stdout[-8000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-4000:])
            LEDGER["12. soap_lora_train"]["status"] = "FAILED"
            raise RuntimeError("SOAP training failed - no adapter saved.")
        SOAP_ADAPTER_DIR = os.path.join(WORKSPACE, "exports", "soap", f"soap-lora-{RUN_ID}")

        # Small deterministic draft run on the preserved TEST rows
        # (drafts only - the test labels remain untouched and unused for training).
        # Run-6 fixes from the 49-min run-5 freeze analysis:
        #   * fp16 compute dtype on the T4 (bf16 falls back to slow emulation),
        #   * max_new_tokens 640 -> 256 (draft rows are short; 640 was pure wall-clock),
        #   * explicit attention_mask into generate,
        #   * greedy config cleanup (do_sample=False with temperature/top_p/top_k
        #     explicitly unset stops the sampler-warnings spam),
        #   * per-row timed [DRAFT i/N] flush prints - 20 silent generations
        #     looked like a frozen cell to the user.
        import json as _json
        import time as _time

        import torch as _t
        from peft import PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        SOAP_META = _json.load(open(os.path.join(SOAP_ADAPTER_DIR, "training_manifest.json"), encoding="utf-8"))
        rows = [ _json.loads(l) for l in open(os.path.join(WORKSPACE, "data", "soap", "soap_test.jsonl"), encoding="utf-8") if l.strip() ]
        # Phase-3 coverage fix (run-8 analysis): draft EVERY preserved test
        # case - run 7 drafted only 20 of 40, which made Stage 13 report
        # missing_drafts for cases that simply had no draft.
        cap_rows = rows[: CONFIG_KNOBS["SOAP_TEST"]]
        tok = AutoTokenizer.from_pretrained(SOAP_ADAPTER_DIR)
        base_model_id, base_rev = SOAP_META.get("base_model_id"), SOAP_META.get("base_model_revision")
        dtype = _t.float16  # T4 Turing: fp16 only - bf16 is emulated/slow (run-5 evidence)
        qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype, bnb_4bit_quant_type="nf4")
        mdl = AutoModelForCausalLM.from_pretrained(base_model_id, revision=base_rev, quantization_config=qcfg, device_map="auto")
        mdl = PeftModel.from_pretrained(mdl, SOAP_ADAPTER_DIR)
        mdl.eval()
        mdl.generation_config.do_sample = False
        mdl.generation_config.temperature = None
        mdl.generation_config.top_p = None
        mdl.generation_config.top_k = None
        sys_msg = ("You draft a structured SOAP note (Subjective, Objective, Assessment, Plan) "
                   "strictly from the supplied consultation text. Never invent vitals, exam "
                   "findings, medications, or doses that are not stated. If information is "
                   "missing, leave the section short ('not stated') instead of filling it.")
        drafts = []
        for i, row in enumerate(cap_rows):
            prompt = tok.apply_chat_template(
                [{"role": "system", "content": sys_msg}, {"role": "user", "content": row["input"]}],
                tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
            t0 = _time.time()
            with _t.no_grad():
                out = mdl.generate(input_ids=prompt, attention_mask=_t.ones_like(prompt),
                                   max_new_tokens=CONFIG_KNOBS["SOAP_MAX_NEW_TOKENS"], do_sample=False,
                                   pad_token_id=tok.pad_token_id or tok.eos_token_id)
            gen = out[0][prompt.shape[1]:]
            text = tok.decode(gen, skip_special_tokens=True).strip()
            drafts.append({"source_row_id": row.get("source_row_id"), "draft_text": text})
            print(f"[DRAFT {i + 1}/{len(cap_rows)}] generated in {_time.time() - t0:.1f}s", flush=True)
        with open(os.path.join(WORKSPACE, "data", "soap", "soap_test_drafts.jsonl"), "w", encoding="utf-8") as fh:
            for d in drafts:
                fh.write(_json.dumps(d, ensure_ascii=False) + "\n")
        print(f"[STAGE-12-OK] drafts generated for {len(drafts)} test rows")
else:
    if not RUN_SOAP:
        LEDGER["12. soap_lora_train"] = {"status": "SKIPPED(toggle)", "error": "RUN_SOAP=False (Colab quota safeguard)"}
        print("[TOGGLE-SKIP] SOAP training skipped (RUN_SOAP=False).")
    else:
        LEDGER["12. soap_lora_train"] = {"status": "SKIPPED(gate)", "error": "SOAP prep missing or gated"}
        print("[GATE-SKIP] SOAP training skipped.")

In [ ]:
# ================================================
# STAGE 13 - evaluate SOAP outputs + manual review sheet (§7.3.13)
# ================================================
drafts_path = os.path.join(WORKSPACE, "data", "soap", "soap_test_drafts.jsonl")
if os.path.exists(drafts_path):
    with stage("13. soap_eval"):
        import subprocess
        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "evaluate_soap.py"),
                              "--cases", os.path.join(WORKSPACE, "data", "soap", "soap_test.jsonl"),
                              "--drafts", drafts_path,
                              "--max-cases", str(CONFIG_KNOBS["SOAP_TEST"]),
                              "--report", os.path.join(WORKSPACE, "reports", f"soap_eval_{RUN_ID}.json"),
                              "--review-sheet", os.path.join(WORKSPACE, "reports", f"soap_review_sheet_{RUN_ID}.csv")],
                             capture_output=True, text=True, timeout=3200)
        print(res.stdout[-7000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-3000:])
            LEDGER["13. soap_eval"]["status"] = "FAILED"
            raise RuntimeError("SOAP evaluation failed.")
        print("[STAGE-13-OK] review sheet is for a VETERINARIAN to fill; automated numbers are not a clinical verdict.")
else:
    LEDGER["13. soap_eval"] = {"status": "NOT RUN", "error": "no drafts generated in stage 12"}
    print("[NOT-RUN] SOAP evaluation skipped (no drafts).")

In [ ]:
# ================================================
# STAGE 14 - optional Roman Urdu alignment experiment, kept SEPARATE (§7.3.14)
# ================================================
# Run-8: enabled by owner mandate. The contract keeps Model C (Urdu-script ->
# Roman-Urdu normalization) a SEPARATE text-to-text task (§5/§12.1): this
# stage PREPARES the aligned pairs and their report - it trains no model and
# touches no audio data. The optional alpaca mix (APPROVED_OPTIONAL) stays
# excluded from the first release.
RUN_ROMAN_STAGE = RUN_MODEL_C
if RUN_ROMAN_STAGE and "Mavkif/Roman-Urdu-Parl-split" in APPROVED_IDS:
    with stage("14. roman_urdu_optional"):
        import shutil
        import subprocess

        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "prepare_roman_urdu_data.py"),
                              "--out-dir", os.path.join(WORKSPACE, "data", "roman_urdu"),
                              "--max-per-split", str(CONFIG_KNOBS["ROMAN_URDU_TRAIN"]),
                              "--max-test-rows", "80",
                              "--seed", str(SEED)],
                             capture_output=True, text=True, timeout=3600)
        print(res.stdout[-6000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-3000:])
            LEDGER["14. roman_urdu_optional"]["status"] = "FAILED"
            raise RuntimeError("Roman Urdu preparation failed.")
        # Keep the experiment's report visible among the run's exported
        # artifacts; the jsonl pairs stay in data/roman_urdu (separate task
        # files - contract forbids JOINING them into ASR/SOAP stores).
        rep_src = os.path.join(WORKSPACE, "data", "roman_urdu", "roman_urdu_prep_report.json")
        if os.path.exists(rep_src):
            shutil.copyfile(rep_src, os.path.join(WORKSPACE, "exports", "roman_urdu_prep_report.json"))
        print("[STAGE-14-OK] aligned pairs prepared; Model C transliteration TRAINING is a separate later stage - never bundled into Whisper and never claimed as an ASR capability.")
elif "RUN_ROMAN_STAGE" in globals() and not RUN_ROMAN_STAGE:
    LEDGER["14. roman_urdu_optional"] = {"status": "SKIPPED(toggle)", "error": "RUN_MODEL_C=False (Colab quota safeguard)"}
    print("[TOGGLE-SKIP] Roman Urdu preparation skipped (RUN_MODEL_C=False).")

In [ ]:
# ================================================
# STAGE 14b - Model C transliteration LoRA training (§5 Model C - SEPARATE task)
# ================================================
if RUN_MODEL_C and LEDGER.get("14. roman_urdu_optional", {}).get("status") == "OK":
    with stage("14b. roman_lora_train", note="mT5-small + LoRA r=16 on q/k/v/o; PURE fp32; targets prefixed '<extra_id_0>' (mt5 span-corruption alignment; decode strips it) - run-11 remediation"):
        import subprocess
        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "train_transliteration_lora.py"),
                              "--train-jsonl", os.path.join(WORKSPACE, "data", "roman_urdu", "roman_urdu_train.jsonl"),
                              "--val-jsonl", os.path.join(WORKSPACE, "data", "roman_urdu", "roman_urdu_validation.jsonl"),
                              "--base-model", "google/mt5-small",
                              "--learning-rate", str(CONFIG_KNOBS["ROMAN_LR"]),
                              "--eval-steps", str(CONFIG_KNOBS["ROMAN_EVAL_STEPS"]),
                              "--epochs", str(CONFIG_KNOBS["ROMAN_EPOCHS"]),
                              "--lora-targets", CONFIG_KNOBS["ROMAN_TARGETS"],
                              "--max-samples", str(CONFIG_KNOBS["ROMAN_URDU_TRAIN"]),
                              "--max-val-samples", str(CONFIG_KNOBS["ROMAN_URDU_VAL"]),
                              "--model-scale", "full",
                              "--export-dir", os.path.join(WORKSPACE, "exports", "roman"),
                              "--out-name", f"roman-lora-{RUN_ID}"],
                             capture_output=True, text=True, timeout=21600)
        print(res.stdout[-8000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-4000:])
            LEDGER["14b. roman_lora_train"]["status"] = "FAILED"
            raise RuntimeError("Model C LoRA training failed - adapter NOT saved.")
        ROMAN_ADAPTER_DIR = os.path.join(WORKSPACE, "exports", "roman", f"roman-lora-{RUN_ID}")
        print("[STAGE-14B-OK] Model C adapter saved; transliteration is SEPARATE from ASR - never bundled into Whisper.")
else:
    LEDGER["14b. roman_lora_train"] = {
        "status": "SKIPPED(toggle)" if not RUN_MODEL_C else "NOT RUN",
        "error": "RUN_MODEL_C=False (quota safeguard)" if not RUN_MODEL_C else "stage 14 prep missing/failed - no pairs to train on",
    }
    print("[NOT-RUN] Model C training skipped (toggle off or prep stage not OK).")

In [ ]:
# ================================================
# STAGE 14c - Model C evaluation: exact match / CER + fluent-review table
# ================================================
roman_adapter = os.path.join(WORKSPACE, "exports", "roman", f"roman-lora-{RUN_ID}")
if RUN_MODEL_C and LEDGER.get("14b. roman_lora_train", {}).get("status") == "OK":
    with stage("14c. roman_eval"):
        import subprocess
        res = subprocess.run([sys.executable, os.path.join(ML, "scripts", "evaluate_transliteration.py"),
                              "--cases", os.path.join(WORKSPACE, "data", "roman_urdu", "roman_urdu_test.jsonl"),
                              "--adapter", roman_adapter,
                              "--base-model", "google/mt5-small",
                              "--max-rows", str(CONFIG_KNOBS["ROMAN_URDU_TEST"]),
                              "--report",
                              os.path.join(WORKSPACE, "exports", "roman", f"roman_eval_{RUN_ID}.json")],
                             capture_output=True, text=True, timeout=7200)
        print(res.stdout[-6000:] if res.stdout else "(no stdout)")
        if res.returncode != 0:
            print(res.stderr[-3000:])
            LEDGER["14c. roman_eval"]["status"] = "FAILED"
            raise RuntimeError("Model C evaluation failed.")
        print("[STAGE-14C-OK] deterministic metrics only; fluent-speaker review table is BLANK and mandatory before reliance.")
else:
    LEDGER["14c. roman_eval"] = {
        "status": "SKIPPED(toggle)" if not RUN_MODEL_C else "NOT RUN",
        "error": "RUN_MODEL_C=False (quota safeguard)" if not RUN_MODEL_C else "stage 14b produced no adapter",
    }
    print("[NOT-RUN] Model C evaluation skipped (toggle off or no adapter).")

In [ ]:
# ================================================
# STAGE 15 - save artifacts, manifests, metrics, model-card notes (§7.3.15)
# ================================================
with stage("15. export_artifacts"):
    import shutil as _sh

    EXPORTS = os.path.join(WORKSPACE, "exports")
    manifest_out = os.path.join(EXPORTS, "run_manifest.json")
    run_manifest = {
        "run_id": RUN_ID,
        "notebook": "voxsoap_ml_training.ipynb",
        "stages": LEDGER,
        "config_knobs": CONFIG_KNOBS,
        "seed": SEED,
        "approved_sources": sorted(APPROVED_IDS) if "APPROVED_IDS" in globals() else [],
        "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
        "claim": "SMOKE-TEST scale only. NOT a claim that models are trained, deployed or veterinary-validated.",
    }
    if "RUNTIME_MANIFEST" in globals():
        run_manifest["dataset_manifest"] = RUNTIME_MANIFEST
    with open(manifest_out, "w", encoding="utf-8") as fh:
        json.dump(run_manifest, fh, indent=2)

    # Model-card notes (honest labeling per §17 rules)
    card = f"""# VoxSOAP ML artifacts ({RUN_ID})

## ASR component (if exported)
- Base: openai/whisper-small (+ exact revision recorded in training_manifest.json)
- Adapter: PEFT LoRA (NOT a complete model on its own)
- Data: capped subsets of license-approved sources only
- NOT veterinary-validated. No claim of clinical accuracy.

## SOAP component (if exported)
- Base model id/revision recorded inside its training_manifest.json
- Objective: SFT with loss ONLY on assistant-response tokens
- Data: Tushar9802/medscribe-soap-712 (HUMAN medical - not veterinary-validated)
- Output remains an assistive DRAFT; the deterministic clinical guard and vet confirmation still rule.

## Roman Urdu component (if exported)
- Model C transliteration (SEPARATE task, never bundled into Whisper, never claimed as ASR)
- Base: google/mt5-small (+ exact revision in its training_manifest.json)
- Pairs: capped Mavkif/Roman-Urdu-Parl-split subsets (parliamentary domain - NOT veterinary speech)
- Deterministic scores are machine-only; the fluent-speaker review table must be filled before any reliance (§4.5)

## Attribution
- google/fleurs used under CC-BY-4.0 - attribution required.

## Privacy
- No real, identifiable clinic recordings or patient records were ingested.
"""
    with open(os.path.join(EXPORTS, "model_card_notes.md"), "w", encoding="utf-8") as fh:
        fh.write(card)

    # Keep every metrics/report file together
    reports_dir = os.path.join(WORKSPACE, "reports")
    for name in os.listdir(reports_dir):
        src, dst = os.path.join(reports_dir, name), os.path.join(EXPORTS, "reports")
        os.makedirs(dst, exist_ok=True)
        _sh.copy2(src, os.path.join(dst, name))
    print("[STAGE-15-OK] exports ready.")

In [ ]:
# ================================================
# STAGE 16 - package and verify the export archive (§7.3.16)
# ================================================
with stage("16. package_export_zip"):
    ARCHIVE = f"/content/voxsoap_ml_exports_{RUN_ID}.zip"
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(os.path.join(WORKSPACE, "exports")):
            for f in files:
                fpath = os.path.join(root, f)
                zf.write(fpath, os.path.relpath(fpath, os.path.join(WORKSPACE, "exports")))
        for root, _, files in os.walk(os.path.join(WORKSPACE, "data", "soap")):
            for f in files:
                if f.endswith(".json") or f.endswith(".jsonl"):
                    fpath = os.path.join(root, f)
                    zf.write(fpath, os.path.relpath(fpath, os.path.join(WORKSPACE, "data", "soap")))
    with zipfile.ZipFile(ARCHIVE) as zf:
        infos = zf.infolist()
        bad = zf.testzip()
        if bad:
            raise RuntimeError(f"zip verification failed on: {bad}")
        total = sum(i.file_size for i in infos)
    print(f"[STAGE-16-OK] archive {ARCHIVE} - entries={len(infos)} uncompressed={round(total/1024/1024, 2)} MB")
    if len(infos) == 0:
        raise RuntimeError("export archive is empty - nothing was successfully saved.")

In [ ]:
# ================================================
# STAGE 17 - print paths/sizes WITHOUT claiming success on failures (§7.3.17)
# ================================================
with stage("17. final_summary"):
    print("\n" + "=" * 76)
    print("STAGE LEDGER - honest outcome of every stage")
    print("=" * 76)
    for name, record in LEDGER.items():
        line = f"  [{record['status']:>12}] {name}"
        if record.get("error"):
            line += f"  ({str(record['error'])[:110]})"
        print(line)
    print("\nEXPORTED FILES AND SIZES:")
    for root, _, files in os.walk(os.path.join(WORKSPACE, "exports")):
        for f in files:
            p = os.path.join(root, f)
            print(f"  {round(os.path.getsize(p)/1024, 1):>12} KB  {os.path.relpath(p, WORKSPACE)}")
    zip_path = globals().get("ARCHIVE")
    if zip_path and os.path.exists(zip_path):
        print(f"\nDOWNLOAD THIS FILE: {zip_path}")
        print("Target on the laptop: D:\\Pawfolio AI\\ml-training\\artifacts\\ (create it first if missing).")

    failed = [n for n, r in LEDGER.items() if r["status"] == "FAILED"]
    skipped = [n for n, r in LEDGER.items() if str(r["status"]).startswith("SKIPPED") or r["status"] == "NOT RUN"]
    ok = [n for n, r in LEDGER.items() if r["status"] == "OK"]
    print("\n" + "=" * 76)
    if failed:
        print(f"RUN INCOMPLETE: {len(failed)} stage(s) FAILED, {len(skipped)} skipped/not-run, {len(ok)} ok.")
        print("DO NOT report metrics or claim training success. Fix the failing stage and re-run.")
    elif len(ok) == 0:
        print("RUN ABORTED: nothing completed. Check earlier errors in this notebook.")
    else:
        print(f"SMOKE RUN COMPLETE: {len(ok)} ok, {len(skipped)} skipped(gate)/not-run.")
        print("This certifies the PIPELINE ONLY - training quality comes from later full runs, real evaluation and the vet benchmark.")
    print("NOTE: this notebook was authored on the feature branch but NOT executed by the coding agent (no Colab access at authoring time) - every status above is produced at your run time.")
    print("Nothing here claims completion of serving or Next.js integration (that is Phase 4/5).");